In [0]:
# ===========================================
# Notebook Name:
# 02_ingest_deck
#
# Purpose:
# Fetch uncollected Pokemon deck HTML pages
# and store the raw responses in Bronze Delta.
#
# Input:
# pokemon.silver.tournament_results
#
# Output:
# pokemon.bronze.deck_raw
# pokemon.bronze.scrape_log
#
# Data quality:
# - deck_code must not be null
# - HTTP response must be successful
# - HTML must contain the deck code
# - identical responses are not duplicated
# ===========================================

In [0]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any
import hashlib
import time
import uuid

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    TimestampType,
)

CATALOG_NAME = "pokemon"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

TOURNAMENT_RESULTS_TABLE = (
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.tournament_results"
)

DECK_RAW_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.deck_raw"
)

SCRAPE_LOG_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.scrape_log"
)

REQUEST_INTERVAL_SECONDS = 1.5
REQUEST_TIMEOUT_SECONDS = 30

print("Input :", TOURNAMENT_RESULTS_TABLE)
print("Output:", DECK_RAW_TABLE)

In [0]:
result_decks_df = (
    spark.table(TOURNAMENT_RESULTS_TABLE)
    .select(
        "deck_code",
        "deck_print_url",
    )
    .filter(F.col("deck_code").isNotNull())
    .filter(F.trim(F.col("deck_code")) != "")
    .dropDuplicates(["deck_code"])
)

display(
    result_decks_df.orderBy("deck_code")
)

print(
    "Unique deck count:",
    result_decks_df.count(),
)

In [0]:
collected_decks_df = (
    spark.table(DECK_RAW_TABLE)
    .select("deck_code")
    .dropDuplicates()
)

pending_decks_df = (
    result_decks_df.alias("source")
    .join(
        collected_decks_df.alias("collected"),
        on="deck_code",
        how="left_anti",
    )
)

display(
    pending_decks_df.orderBy("deck_code")
)

pending_count = pending_decks_df.count()

print("Pending deck count:", pending_count)

In [0]:
pending_decks = [
    row.asDict()
    for row in pending_decks_df.collect()
]

print("Decks to fetch:", len(pending_decks))

for deck in pending_decks[:5]:
    print(deck)

In [0]:
REQUEST_HEADERS = {
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
    "User-Agent": (
        "Mozilla/5.0 "
        "(compatible; PokemonLakehouse/0.1)"
    ),
}


def fetch_deck_html(
    deck_code: str,
    deck_print_url: str,
    timeout: int = REQUEST_TIMEOUT_SECONDS,
) -> dict[str, Any]:

    started_at = time.perf_counter()
    scraped_at = datetime.now(timezone.utc)

    response = requests.get(
        deck_print_url,
        headers=REQUEST_HEADERS,
        timeout=timeout,
    )

    elapsed_ms = int(
        (time.perf_counter() - started_at) * 1000
    )

    response.raise_for_status()

    response.encoding = (
        response.apparent_encoding
        or response.encoding
        or "utf-8"
    )

    html = response.text

    if not html.strip():
        raise ValueError(
            f"Empty HTML returned for deck {deck_code}"
        )

    if deck_code not in html:
        raise ValueError(
            f"Deck code was not found in HTML: {deck_code}"
        )

    response_hash = hashlib.sha256(
        html.encode("utf-8")
    ).hexdigest()

    return {
        "deck_code": deck_code,
        "source_url": response.url,
        "http_status": response.status_code,
        "payload": html,
        "payload_format": "html",
        "response_hash": response_hash,
        "scraped_at": scraped_at,
        "elapsed_ms": elapsed_ms,
    }

In [0]:
if pending_decks:
    test_deck = pending_decks[0]

    test_result = fetch_deck_html(
        deck_code=test_deck["deck_code"],
        deck_print_url=test_deck["deck_print_url"],
    )

    print("deck_code:", test_result["deck_code"])
    print("status:", test_result["http_status"])
    print("HTML length:", len(test_result["payload"]))
    print("hash:", test_result["response_hash"])
else:
    print("No pending decks.")

In [0]:
raw_rows: list[dict[str, Any]] = []
log_rows: list[dict[str, Any]] = []

total = len(pending_decks)

for index, deck in enumerate(
    pending_decks,
    start=1,
):
    deck_code = deck["deck_code"]
    deck_print_url = deck["deck_print_url"]
    run_id = str(uuid.uuid4())
    execution_time = datetime.now(timezone.utc)

    print(
        f"[{index}/{total}] Fetching {deck_code}"
    )

    try:
        result = fetch_deck_html(
            deck_code=deck_code,
            deck_print_url=deck_print_url,
        )

        raw_rows.append({
            "deck_code": result["deck_code"],
            "source_url": result["source_url"],
            "http_status": result["http_status"],
            "payload": result["payload"],
            "payload_format": result["payload_format"],
            "response_hash": result["response_hash"],
            "scraped_at": result["scraped_at"],
        })

        log_rows.append({
            "run_id": run_id,
            "source_type": "deck",
            "source_id": deck_code,
            "request_url": result["source_url"],
            "http_status": result["http_status"],
            "status": "SUCCESS_INSERTED",
            "elapsed_ms": result["elapsed_ms"],
            "records_found": 1,
            "error_type": None,
            "error_message": None,
            "scraped_at": result["scraped_at"],
        })

        print(
            f"  Success: {len(result['payload'])} bytes"
        )

    except Exception as error:
        log_rows.append({
            "run_id": run_id,
            "source_type": "deck",
            "source_id": deck_code,
            "request_url": deck_print_url,
            "http_status": None,
            "status": "FAILED",
            "elapsed_ms": None,
            "records_found": 0,
            "error_type": type(error).__name__,
            "error_message": str(error)[:2000],
            "scraped_at": execution_time,
        })

        print(f"  Failed: {error}")

    if index < total:
        time.sleep(REQUEST_INTERVAL_SECONDS)

print("Success count:", len(raw_rows))
print(
    "Failure count:",
    sum(row["status"] == "FAILED" for row in log_rows),
)

In [0]:
deck_raw_schema = StructType([
    StructField("deck_code", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("payload", StringType(), False),
    StructField("payload_format", StringType(), False),
    StructField("response_hash", StringType(), False),
    StructField("scraped_at", TimestampType(), False),
])

In [0]:
if raw_rows:
    deck_raw_df = (
        spark.createDataFrame(
            raw_rows,
            schema=deck_raw_schema,
        )
        .withColumn(
            "ingestion_date",
            F.to_date("scraped_at"),
        )
    )

    display(
        deck_raw_df.select(
            "deck_code",
            "http_status",
            F.length("payload").alias("payload_length"),
            "response_hash",
            "scraped_at",
        )
    )
else:
    deck_raw_df = None
    print("No successful deck responses.")

In [0]:
if deck_raw_df is not None:
    (
        deck_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(DECK_RAW_TABLE)
    )

    print(
        f"{deck_raw_df.count()} rows inserted "
        f"into {DECK_RAW_TABLE}"
    )
else:
    print("No rows were inserted.")

In [0]:
scrape_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("request_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("elapsed_ms", LongType(), True),
    StructField("records_found", IntegerType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("scraped_at", TimestampType(), False),
])

In [0]:
if log_rows:
    scrape_log_df = (
        spark.createDataFrame(
            log_rows,
            schema=scrape_log_schema,
        )
        .withColumn(
            "ingestion_date",
            F.to_date("scraped_at"),
        )
    )

    (
        scrape_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SCRAPE_LOG_TABLE)
    )

    display(scrape_log_df)
else:
    print("No scrape logs to write.")

In [0]:
display(
    spark.table(DECK_RAW_TABLE)
    .select(
        "deck_code",
        "http_status",
        "payload_format",
        F.length("payload").alias("payload_length"),
        "response_hash",
        "scraped_at",
    )
    .orderBy(F.col("scraped_at").desc())
)

In [0]:
display(
    spark.table(DECK_RAW_TABLE)
    .agg(
        F.count("*").alias("raw_response_count"),
        F.countDistinct("deck_code").alias(
            "unique_deck_count"
        ),
        F.min(F.length("payload")).alias(
            "minimum_payload_length"
        ),
        F.max(F.length("payload")).alias(
            "maximum_payload_length"
        ),
    )
)

In [0]:
display(
    spark.table(SCRAPE_LOG_TABLE)
    .filter(F.col("source_type") == "deck")
    .orderBy(F.col("scraped_at").desc())
)